# Calculate distance to nearest AED, Ambulance, MUG and PIT

In [ ]:
import pandas as pd
import geopandas as gpd
from scipy.spatial import KDTree
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Importing Datasets

In [ ]:
# setting parameters for Papermill
input_data_path = 'Results/total_df.csv'
url2 = 'https://raw.githubusercontent.com/JeroenGuillierme/Project-MDA/main/Data/'
output_data_path = 'Results/preprocessed_data_with_distances.csv'

In [ ]:
data = pd.read_csv(input_data_path) # load data from locally stored directory

# Load Belgium with regions shapefile
belgium_with_provinces_boundary = gpd.read_file(f'{url2}BELGIUM_-_Provinces.geojson')

# Ensure the CRS (Coordinate Reference System) is set to WGS84 (latitude/longitude)
belgium_with_provinces_boundary = belgium_with_provinces_boundary.to_crs(epsg=4326)

pd.set_option('display.max_columns', None)

## 2. Functions

Here a custom function is imported from the Functions.py script, which will be used further in this Notebook for calculating the distances.

In [ ]:
from Functions import haversine

## 3. Prepare datasets for distance calculations

In [ ]:
# Ensure Latitude and Longitude are not missing
data = data.dropna(subset=['Latitude', 'Longitude'])

# Add new columns
data['distance_to_aed'] = 0.0
data['distance_to_ambulance'] = 0.0
data['distance_to_mug'] = 0.0
data['distance_to_pit'] = 0.0

# Filter for necessary locations
aed_locations = data[data['AED'] == 1]
ambulance_locations = data[data['Ambulance'] == 1]
mug_locations = data[data['Mug'] == 1]
pit_locations = data[data['PIT'] == 1]
intervention_locations = data[data['Intervention'] == 1]

print(len(intervention_locations), len(aed_locations), len(ambulance_locations), len(mug_locations), len(pit_locations))

## 4. Create KDTree objects

In [ ]:
# Create KDTree objects
if not aed_locations.empty:
    tree_aeds = KDTree(aed_locations[['Latitude', 'Longitude']])
if not ambulance_locations.empty:
    tree_ambulances = KDTree(ambulance_locations[['Latitude', 'Longitude']])
if not mug_locations.empty:
    tree_mugs = KDTree(mug_locations[['Latitude', 'Longitude']])
if not pit_locations.empty:
    tree_pits = KDTree(pit_locations[['Latitude', 'Longitude']])

## 5. Calculate distances for each intervention to nearest AED, Ambulance, Mug and PIT

In [ ]:
# Calculate distances
for idx, intervention_point in intervention_locations.iterrows():
    coords = intervention_point[['Latitude', 'Longitude']].values
    distance_to_mug, _ = tree_mugs.query(coords)
    distance_to_ambulance, _ = tree_ambulances.query(coords)
    distance_to_aed, _ = tree_aeds.query(coords)
    distance_to_pit, _ = tree_pits.query(coords)
    
    # Get the nearest AED, Ambulance, and Mug coordinates
    nearest_aed_idx = tree_aeds.query(coords, k=1)[1]
    nearest_ambulance_idx = tree_ambulances.query(coords, k=1)[1]
    nearest_mug_idx = tree_mugs.query(coords, k=1)[1]
    nearest_pit_idx = tree_pits.query(coords, k=1)[1]
    
    nearest_aed = aed_locations.iloc[nearest_aed_idx]
    nearest_ambulance = ambulance_locations.iloc[nearest_ambulance_idx]
    nearest_mug = mug_locations.iloc[nearest_mug_idx]
    nearest_pit = pit_locations.iloc[nearest_pit_idx]
    
    intervention_locations.at[idx, 'distance_to_aed'] = haversine(
        intervention_point['Longitude'], intervention_point['Latitude'],
        nearest_aed['Longitude'], nearest_aed['Latitude']
    )
    intervention_locations.at[idx, 'distance_to_ambulance'] = haversine(
        intervention_point['Longitude'], intervention_point['Latitude'],
        nearest_ambulance['Longitude'], nearest_ambulance['Latitude']
    )
    intervention_locations.at[idx, 'distance_to_mug'] = haversine(
        intervention_point['Longitude'], intervention_point['Latitude'],
        nearest_mug['Longitude'], nearest_mug['Latitude']
    )
    intervention_locations.at[idx, 'distance_to_pit'] = haversine(
        intervention_point['Longitude'], intervention_point['Latitude'],
        nearest_pit['Longitude'], nearest_pit['Latitude']
    )

### 5.1 Complete new columns in dataset for the aed, ambulance, mug and pit locations

In [ ]:
# Filling in new columns for AED locations
aed_locations.loc[:, 'distance_to_aed'] = 0
aed_locations.loc[:, 'distance_to_ambulance'] = np.nan
aed_locations.loc[:, 'distance_to_mug'] = np.nan
aed_locations.loc[:, 'distance_to_pit'] = np.nan

# Filling in new columns for mug locations
mug_locations.loc[:, 'distance_to_aed'] = np.nan
mug_locations.loc[:, 'distance_to_ambulance'] = np.nan
mug_locations.loc[:, 'distance_to_mug'] = 0
mug_locations.loc[:, 'distance_to_pit'] = np.nan

# Filling in new columns for ambulance locations
ambulance_locations.loc[:, 'distance_to_aed'] = np.nan
ambulance_locations.loc[:, 'distance_to_ambulance'] = 0
ambulance_locations.loc[:, 'distance_to_mug'] = np.nan
ambulance_locations.loc[:, 'distance_to_pit'] = np.nan

# Filling in new columns for pit locations
pit_locations.loc[:, 'distance_to_aed'] = np.nan
pit_locations.loc[:, 'distance_to_ambulance'] = np.nan
pit_locations.loc[:, 'distance_to_mug'] = np.nan
pit_locations.loc[:, 'distance_to_pit'] = 0

### 5.2 Concatenate all datasets back together

In [ ]:
# Concatente data back together with distances
result = pd.concat([intervention_locations, aed_locations, mug_locations, ambulance_locations, pit_locations], axis=0)
# Reset index
result.reset_index(drop=True, inplace=True)

In [ ]:
result.head(10)

In [ ]:
print('Missing values per variable: \n', result.isnull().sum())

## 6. Data visualisations

### 6.1 Histograms for visualisation of distributions

In [ ]:
# Check for missing values
interventions_data = result[result['Intervention']==1]
print('Missing values per variable: \n', interventions_data.isnull().sum())
print('Length dataset: ', len(interventions_data))

# Setting the style for the plots
sns.set(style="whitegrid")

# Create a figure and a grid of subplots
fig, axes = plt.subplots(3, 2, figsize=(15, 18))

# Scatterplot coordinates
sns.scatterplot(data=interventions_data, x='Longitude', y='Latitude', hue='Province', ax=axes[0,0])
axes[0, 0].set_title('Coordinate pairs of all interventions')

# Plot histogram for response times
sns.histplot(interventions_data['T3-T0'], bins=100, kde=True, log_scale=True, ax=axes[0, 1])
axes[0, 1].set_title('Log Distribution of Response Times')
axes[0, 1].set_xlabel('Log Response Time (minutes)')
axes[0, 1].set_ylabel('Frequency')

# Plot histogram for distance to the closest AED
sns.histplot(interventions_data['distance_to_aed'], bins=100, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Distances to the Closest AED')
axes[1, 0].set_xlabel('Distance to closest AED')
axes[1, 0].set_ylabel('Frequency')

# Plot histogram for distance to the closest ambulance
sns.histplot(interventions_data['distance_to_ambulance'], bins=100, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribution of Distances to the Closest Ambulance Location')
axes[1, 1].set_xlabel('Distance to closest Ambulance location')
axes[1, 1].set_ylabel('Frequency')

# Plot histogram for distance to the closest Mug
sns.histplot(interventions_data['distance_to_mug'], bins=100, kde=True, ax=axes[2, 0])
axes[2, 0].set_title('Distribution of Distances to the Closest Mug Location')
axes[2, 0].set_xlabel('Distance to closest Mug location')
axes[2, 0].set_ylabel('Frequency')

# Plot histogram for distance to the closest PIT
sns.histplot(interventions_data['distance_to_pit'], bins=100, kde=True, ax=axes[2, 1])
axes[2, 1].set_title('Distribution of Distances to the Closest PIT Location')
axes[2, 1].set_xlabel('Distance to closest PIT location')
axes[2, 1].set_ylabel('Frequency')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()

## 7. Save new dataset to the repository

In [ ]:
result.to_csv(output_data_path, index=False)